# Direct vs Chain-of-Thought (CoT) Evaluation

In this notebook, we compare the performance of Direct prompting (Zero-Shot) versus Chain-of-Thought (CoT) prompting for Yelp review classification.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from tqdm import tqdm

from src.config import (
    PREDICTIONS_DIR,
    PROMPT_SUBSET_FILE,
)

from src.prompts import build_direct_prompt, build_cot_prompt
from src.llm_runner import call_ollama, extract_json_block, DEFAULT_MODEL
from src.evaluation import compute_metrics, print_metrics

In [2]:
import json

def parse_direct_or_cot_prediction(raw_output: str):
    parsed = {
        "raw_output": raw_output,
        "json_valid": False,
        "stars_pred": None,
        "reasoning_pred": None,
        "explanation_pred": None,
        "parse_error": None,
    }

    try:
        json_text = extract_json_block(raw_output)
        if json_text is None:
            parsed["parse_error"] = "No JSON object found"
            return parsed

        obj = json.loads(json_text)

        stars = obj.get("stars", None)
        explanation = obj.get("explanation", None)
        reasoning = obj.get("reasoning", None)

        if not isinstance(stars, int):
            parsed["parse_error"] = "stars is not an integer"
            return parsed

        if stars < 1 or stars > 5:
            parsed["parse_error"] = "stars out of range"
            return parsed

        if explanation is not None and not isinstance(explanation, str):
            parsed["parse_error"] = "explanation is not a string"
            return parsed

        if reasoning is not None and not isinstance(reasoning, str):
            parsed["parse_error"] = "reasoning is not a string"
            return parsed

        parsed["json_valid"] = True
        parsed["stars_pred"] = stars
        parsed["reasoning_pred"] = reasoning.strip() if isinstance(reasoning, str) else None
        parsed["explanation_pred"] = explanation.strip() if isinstance(explanation, str) else None
        return parsed

    except Exception as e:
        parsed["parse_error"] = str(e)
        return parsed

In [3]:
df = pd.read_csv(PROMPT_SUBSET_FILE)
print(df.shape)
df.head()

(500, 5)


,text,label_raw,stars,char_length,word_length
0,My family and I tried this for the first time ...,3,4,327,63
1,"If I were playing the word association game, a...",0,1,1809,334
2,I think this would have been a five star if we...,3,4,298,55
3,I prefer the El Hefe on Mill for having more s...,1,2,762,135
4,I just moved into the neighborhood and was try...,1,2,687,129


In [4]:
sample_df = df.head(5).copy()
sample_df[["text", "stars"]]

,text,stars
0,My family and I tried this for the first time ...,4
1,"If I were playing the word association game, a...",1
2,I think this would have been a five star if we...,4
3,I prefer the El Hefe on Mill for having more s...,2
4,I just moved into the neighborhood and was try...,2


In [5]:
def run_direct_inference(input_df, model_name=DEFAULT_MODEL):
    rows = []

    for _, row in tqdm(input_df.iterrows(), total=len(input_df)):
        review_text = row["text"]
        true_stars = int(row["stars"])

        prompt = build_direct_prompt(review_text)
        raw_output = call_ollama(prompt=prompt, model=model_name, temperature=0.0)
        parsed = parse_direct_or_cot_prediction(raw_output)

        rows.append({
            "text": review_text,
            "stars": true_stars,
            "prompt_type": "direct",
            **parsed
        })

    return pd.DataFrame(rows)

In [6]:
def run_cot_inference(input_df, model_name=DEFAULT_MODEL):
    rows = []

    for _, row in tqdm(input_df.iterrows(), total=len(input_df)):
        review_text = row["text"]
        true_stars = int(row["stars"])

        prompt = build_cot_prompt(review_text)
        raw_output = call_ollama(prompt=prompt, model=model_name, temperature=0.0)
        parsed = parse_direct_or_cot_prediction(raw_output)

        rows.append({
            "text": review_text,
            "stars": true_stars,
            "prompt_type": "cot",
            **parsed
        })

    return pd.DataFrame(rows)

In [7]:
direct_test = run_direct_inference(sample_df, model_name=DEFAULT_MODEL)
direct_test[["stars", "stars_pred", "json_valid", "parse_error", "explanation_pred"]]

100%|███████████████████████████████████████████████████████████| 5/5 [00:17<00:00,  3.54s/it]


,stars,stars_pred,json_valid,parse_error,explanation_pred
0,4,4,True,None,The reviewer enjoyed the unique food and drink...
1,1,2,True,None,The reviewer had a disappointing experience wi...
2,4,3,True,None,"The reviewer had mixed experience, food was ju..."
3,2,2,True,None,The reviewer had a negative experience with th...
4,2,1,True,None,The reviewer had a negative experience with th...


In [8]:
cot_test = run_cot_inference(sample_df, model_name=DEFAULT_MODEL)
cot_test[["stars", "stars_pred", "json_valid", "parse_error", "reasoning_pred", "explanation_pred"]]

100%|███████████████████████████████████████████████████████████| 5/5 [00:07<00:00,  1.51s/it]


,stars,stars_pred,json_valid,parse_error,reasoning_pred,explanation_pred
0,4,4,True,None,The reviewer enjoyed the unique food and drink...,Generally positive experience
1,1,1,True,None,The reviewer expresses disappointment and a ne...,Extremely negative review
2,4,3,True,None,"The reviewer had mixed experience, food was ju...",Mixed experience
3,2,2,True,None,The reviewer expresses dissatisfaction with th...,Negative experience with service and atmosphere
4,2,1,True,None,The reviewer had a negative experience due to ...,Poor customer service


In [9]:
direct_metrics_small = compute_metrics(direct_test)
cot_metrics_small = compute_metrics(cot_test)

print_metrics("DIRECT SMALL TEST", direct_metrics_small)
print_metrics("COT SMALL TEST", cot_metrics_small)


=== DIRECT SMALL TEST ===
total_rows: 5
json_compliance_rate: 1.0
valid_prediction_rows: 5
accuracy: 0.4
macro_f1: 0.2917

=== COT SMALL TEST ===
total_rows: 5
json_compliance_rate: 1.0
valid_prediction_rows: 5
accuracy: 0.6
macro_f1: 0.5


In [10]:
mini_df = df.head(50).copy()

direct_mini = run_direct_inference(mini_df, model_name=DEFAULT_MODEL)
cot_mini = run_cot_inference(mini_df, model_name=DEFAULT_MODEL)

direct_mini_metrics = compute_metrics(direct_mini)
cot_mini_metrics = compute_metrics(cot_mini)

print_metrics("DIRECT 50 ROWS", direct_mini_metrics)
print_metrics("COT 50 ROWS", cot_mini_metrics)

100%|█████████████████████████████████████████████████████████| 50/50 [01:28<00:00,  1.78s/it]


=== DIRECT 50 ROWS ===
total_rows: 50
json_compliance_rate: 1.0
valid_prediction_rows: 50
accuracy: 0.66
macro_f1: 0.5735

=== COT 50 ROWS ===
total_rows: 50
json_compliance_rate: 1.0
valid_prediction_rows: 50
accuracy: 0.72
macro_f1: 0.6039


In [11]:
print("Direct invalid JSON:", (direct_mini["json_valid"] == False).sum())
print("CoT invalid JSON   :", (cot_mini["json_valid"] == False).sum())

Direct invalid JSON: 0
CoT invalid JSON   : 0


In [12]:
direct_df = run_direct_inference(df, model_name=DEFAULT_MODEL)
cot_df = run_cot_inference(df, model_name=DEFAULT_MODEL)

100%|███████████████████████████████████████████████████████| 500/500 [13:32<00:00,  1.62s/it]


In [13]:
direct_path = PREDICTIONS_DIR / "direct_predictions.csv"
cot_path = PREDICTIONS_DIR / "cot_predictions.csv"

direct_df.to_csv(direct_path, index=False)
cot_df.to_csv(cot_path, index=False)

print("Saved:", direct_path)
print("Saved:", cot_path)

Saved: /home/rehan/Documents/yelp-ai-assignment/outputs/predictions/direct_predictions.csv
Saved: /home/rehan/Documents/yelp-ai-assignment/outputs/predictions/cot_predictions.csv


In [14]:
direct_metrics = compute_metrics(direct_df)
cot_metrics = compute_metrics(cot_df)

print_metrics("DIRECT FULL", direct_metrics)
print_metrics("COT FULL", cot_metrics)


=== DIRECT FULL ===
total_rows: 500
json_compliance_rate: 1.0
valid_prediction_rows: 500
accuracy: 0.692
macro_f1: 0.6865

=== COT FULL ===
total_rows: 500
json_compliance_rate: 1.0
valid_prediction_rows: 500
accuracy: 0.678
macro_f1: 0.6591


In [15]:
comparison_df = pd.DataFrame([
    {"method": "direct", **direct_metrics},
    {"method": "cot", **cot_metrics},
])

comparison_df

,method,total_rows,json_compliance_rate,valid_prediction_rows,accuracy,macro_f1
0,direct,500,1.0,500,0.692,0.6865
1,cot,500,1.0,500,0.678,0.6591


In [16]:
direct_failures = direct_df[
    (direct_df["json_valid"] == False) |
    (direct_df["stars"] != direct_df["stars_pred"])
].copy()

cot_failures = cot_df[
    (cot_df["json_valid"] == False) |
    (cot_df["stars"] != cot_df["stars_pred"])
].copy()

print("Direct failures:", len(direct_failures))
print("CoT failures   :", len(cot_failures))

Direct failures: 154
CoT failures   : 161


In [17]:
direct_failures[["text", "stars", "stars_pred", "parse_error", "explanation_pred"]].head(10)

,text,stars,stars_pred,parse_error,explanation_pred
1,"If I were playing the word association game, a...",1,2,None,The reviewer had a disappointing experience at...
2,I think this would have been a five star if we...,4,3,None,"The reviewer had mixed experience, food was de..."
4,I just moved into the neighborhood and was try...,2,1,None,The reviewer had a very negative experience wi...
5,"Went for dim sum. It was moderately busy, but...",4,3,None,The reviewer had an average experience with de...
6,I've been here now about 5-6 times now and am ...,4,5,None,The reviewer provides glowing feedback about t...
7,"Ling & Louie, hmmm, I'm having a bit of hard t...",2,3,None,The reviewer seems neutral about the restauran...
9,Excellent breakfast and a fun place to go with...,5,4,None,"The reviewer uses positive language, mentionin..."
10,There was a wendys. Thats it in my terminal wo...,1,2,None,Brief mention with a negative aspect
11,This was a great disappointment. My boyfriend ...,2,1,None,The reviewer expresses disappointment and frus...
16,i'm sooo glad we didn't have a lot time in her...,5,3,None,"The reviewer had mixed feelings, appreciating ..."


In [18]:
cot_failures[["text", "stars", "stars_pred", "parse_error", "reasoning_pred", "explanation_pred"]].head(10)

,text,stars,stars_pred,parse_error,reasoning_pred,explanation_pred
2,I think this would have been a five star if we...,4,3,None,"The reviewer had mixed experience, food was ju...",Mixed experience
4,I just moved into the neighborhood and was try...,2,1,None,The reviewer had a negative experience due to ...,Poor customer service
5,"Went for dim sum. It was moderately busy, but...",4,3,None,The review mentions that the food was 'decent'...,Neutral experience
6,I've been here now about 5-6 times now and am ...,4,5,None,The reviewer provides detailed and overwhelmin...,The reviewer's glowing praise and lack of sign...
7,"Ling & Louie, hmmm, I'm having a bit of hard t...",2,3,None,The reviewer is conflicted about the restauran...,Mixed experience
10,There was a wendys. Thats it in my terminal wo...,1,2,None,The reviewer mentions a negative aspect of the...,Subpar experience
11,This was a great disappointment. My boyfriend ...,2,1,None,The reviewer expresses disappointment and frus...,Poor service and food quality led to a very ne...
15,HATE Nothing.\n\nDISLIKE Nothing.\n\nLIKE T...,4,5,None,The reviewer has listed multiple positive aspe...,Perfect experience
16,i'm sooo glad we didn't have a lot time in her...,5,4,None,"The reviewer expresses mixed feelings, initial...","Generally positive, but slight disappointment"
23,I took my 5 year old daughter and my father in...,4,5,None,The reviewer had a very positive experience wi...,Exceptional show with impressive sets and chor...


In [19]:
cot_df[["text", "stars", "stars_pred", "reasoning_pred", "explanation_pred"]].head(20)

,text,stars,stars_pred,reasoning_pred,explanation_pred
0,My family and I tried this for the first time ...,4,4,The reviewer enjoyed the unique food and drink...,Generally positive experience
1,"If I were playing the word association game, a...",1,1,The reviewer expresses disappointment and a ne...,Extremely negative review
2,I think this would have been a five star if we...,4,3,"The reviewer had mixed experience, food was ju...",Mixed experience
3,I prefer the El Hefe on Mill for having more s...,2,2,The reviewer expresses dissatisfaction with th...,Negative experience with service and atmosphere
4,I just moved into the neighborhood and was try...,2,1,The reviewer had a negative experience due to ...,Poor customer service
5,"Went for dim sum. It was moderately busy, but...",4,3,The review mentions that the food was 'decent'...,Neutral experience
6,I've been here now about 5-6 times now and am ...,4,5,The reviewer provides detailed and overwhelmin...,The reviewer's glowing praise and lack of sign...
7,"Ling & Louie, hmmm, I'm having a bit of hard t...",2,3,The reviewer is conflicted about the restauran...,Mixed experience
8,I've been a valued Apple customer for nearly t...,1,1,The customer expresses frustration with their ...,Extremely dissatisfied
9,Excellent breakfast and a fun place to go with...,5,5,The reviewer uses positive language throughout...,Very positive review


In [20]:
cot_wrong = cot_df[cot_df["stars"] != cot_df["stars_pred"]].copy()
cot_wrong[["text", "stars", "stars_pred", "reasoning_pred", "explanation_pred"]].head(15)

,text,stars,stars_pred,reasoning_pred,explanation_pred
2,I think this would have been a five star if we...,4,3,"The reviewer had mixed experience, food was ju...",Mixed experience
4,I just moved into the neighborhood and was try...,2,1,The reviewer had a negative experience due to ...,Poor customer service
5,"Went for dim sum. It was moderately busy, but...",4,3,The review mentions that the food was 'decent'...,Neutral experience
6,I've been here now about 5-6 times now and am ...,4,5,The reviewer provides detailed and overwhelmin...,The reviewer's glowing praise and lack of sign...
7,"Ling & Louie, hmmm, I'm having a bit of hard t...",2,3,The reviewer is conflicted about the restauran...,Mixed experience
10,There was a wendys. Thats it in my terminal wo...,1,2,The reviewer mentions a negative aspect of the...,Subpar experience
11,This was a great disappointment. My boyfriend ...,2,1,The reviewer expresses disappointment and frus...,Poor service and food quality led to a very ne...
15,HATE Nothing.\n\nDISLIKE Nothing.\n\nLIKE T...,4,5,The reviewer has listed multiple positive aspe...,Perfect experience
16,i'm sooo glad we didn't have a lot time in her...,5,4,"The reviewer expresses mixed feelings, initial...","Generally positive, but slight disappointment"
23,I took my 5 year old daughter and my father in...,4,5,The reviewer had a very positive experience wi...,Exceptional show with impressive sets and chor...


# Final Observations

Direct prompting achieved **69.2% accuracy**, while Chain-of-Thought prompting achieved **67.8% accuracy** on the Yelp review classification task.

This indicates that **Direct prompting performed better** for this model and dataset. A likely reason is that CoT caused the model to **overthink** relatively simple classification decisions, especially for mixed or ambiguous reviews.

In this experiment, explicit reasoning did not improve classification accuracy and may have reduced decision stability. This suggests that for small, structured label spaces like 1–5 star prediction, **simpler prompting can outperform reasoning-heavy prompting**.